El objetivo de este análisis es construir un modelo de regresión capaz de estimar el precio del alojamiento turístico a partir de variables geográficas, temporales y del tipo de hospedaje. Para ello, se parte del dataset de alojamientos, se transforma a formato largo, se realiza feature engineering, se comparan distintos algoritmos de regresión y finalmente se simula un presupuesto de viaje.

In [48]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from datetime import datetime, timedelta

BASE_DIR = Path.cwd().resolve().parents[1]

OUTPUTS_DIR = BASE_DIR / "outputs" / "regresion_precios"
MODEL_DIR = OUTPUTS_DIR / "modelos"
ML_DIR = BASE_DIR / "data" / "Machine Learning"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(BASE_DIR)
print(OUTPUTS_DIR)
print(MODEL_DIR)
print(ML_DIR)
print((MODEL_DIR / "random_forest_precio.pkl").exists())


C:\Users\Andrea Morales Vega\Downloads\TFM\CulturaTrip_TFM
C:\Users\Andrea Morales Vega\Downloads\TFM\CulturaTrip_TFM\outputs\regresion_precios
C:\Users\Andrea Morales Vega\Downloads\TFM\CulturaTrip_TFM\outputs\regresion_precios\modelos
C:\Users\Andrea Morales Vega\Downloads\TFM\CulturaTrip_TFM\data\Machine Learning
True


En esta etapa se transforma el dataset de alojamientos desde formato ancho a formato largo, con el fin de disponer de una sola variable objetivo de precio y una variable indicadora del tipo de día.

In [49]:
input_path = BASE_DIR / "data" / "clean" / "df_alojamientos.csv"
output_path = ML_DIR / "df_precios_largo.csv"

df = pd.read_csv(input_path)

print("Dimensiones originales:", df.shape)
df.head()

Dimensiones originales: (10280, 15)


,id_alojamiento,id_pais,id_ccaa,id_provincia,mes,categoria_alojamiento,periodo_antelacion,precio_checkin_entre_semana,precio_checkin_fin_semana,tiene_valoraciones,fuente,granularidad_origen,es_dato_replicado,nivel_geografico,valoraciones_norm
0,1,ES,1,4,1,hotel 3 estrellas,1 mes,70.00,72.00,True,hotel_dataestur,provincia,False,provincia,3.7595
1,2,ES,1,4,1,hotel 3 estrellas,1 semana,71.00,73.67,True,hotel_dataestur,provincia,False,provincia,3.7595
2,3,ES,1,4,1,hotel 3 estrellas,2 semanas,71.00,73.67,True,hotel_dataestur,provincia,False,provincia,3.7595
3,4,ES,1,4,1,hotel 3 estrellas,3 meses,67.33,70.33,True,hotel_dataestur,provincia,False,provincia,3.7595
4,5,ES,1,4,1,hotel 4 estrellas,1 mes,75.33,78.67,True,hotel_dataestur,provincia,False,provincia,3.2400


In [50]:
df_largo = df.melt(
    id_vars=[
        "id_alojamiento",
        "id_pais",
        "id_ccaa",
        "id_provincia",
        "mes",
        "categoria_alojamiento",
        "periodo_antelacion",
        "tiene_valoraciones",
        "fuente",
        "granularidad_origen",
        "es_dato_replicado",
        "nivel_geografico",
        "valoraciones_norm"
    ],
    value_vars=[
        "precio_checkin_entre_semana",
        "precio_checkin_fin_semana"
    ],
    var_name="tipo_dia_original",
    value_name="precio"
)

map_tipo_dia = {
    "precio_checkin_entre_semana": "semana",
    "precio_checkin_fin_semana": "fin_semana"
}

df_largo["tipo_dia"] = df_largo["tipo_dia_original"].map(map_tipo_dia)

columnas_finales = [
    "id_alojamiento",
    "id_pais",
    "id_ccaa",
    "id_provincia",
    "mes",
    "categoria_alojamiento",
    "periodo_antelacion",
    "valoraciones_norm",
    "tiene_valoraciones",
    "tipo_dia",
    "precio"
]

df_largo = df_largo[columnas_finales].copy()
df_largo.to_csv(output_path, index=False, encoding="utf-8")

print("Dimensiones formato largo:", df_largo.shape)
display(df_largo.head())
display(df_largo["tipo_dia"].value_counts())

Dimensiones formato largo: (20560, 11)


,id_alojamiento,id_pais,id_ccaa,id_provincia,mes,categoria_alojamiento,periodo_antelacion,valoraciones_norm,tiene_valoraciones,tipo_dia,precio
0,1,ES,1,4,1,hotel 3 estrellas,1 mes,3.7595,True,semana,70.00
1,2,ES,1,4,1,hotel 3 estrellas,1 semana,3.7595,True,semana,71.00
2,3,ES,1,4,1,hotel 3 estrellas,2 semanas,3.7595,True,semana,71.00
3,4,ES,1,4,1,hotel 3 estrellas,3 meses,3.7595,True,semana,67.33
4,5,ES,1,4,1,hotel 4 estrellas,1 mes,3.2400,True,semana,75.33


tipo_dia
semana        10280
fin_semana    10280
Name: count, dtype: int64

La transformación a formato largo permite modelar el precio como una única variable objetivo, diferenciando si el valor corresponde a entre semana o fin de semana. Esto facilita el entrenamiento de modelos de regresión y mejora la consistencia del pipeline.

En esta fase se generan variables derivadas y codificadas para preparar el dataset de entrada al modelo.

In [51]:
output_features_path = ML_DIR / "df_precios_features.csv"

df = df_largo.copy()

def clasificar_temporada(mes):
    if mes in [12, 1, 2]:
        return "baja"
    elif mes in [3, 4, 5, 10, 11]:
        return "media"
    elif mes in [6, 7, 8, 9]:
        return "alta"
    else:
        return "desconocida"

df["temporada"] = df["mes"].apply(clasificar_temporada)

map_tipo_dia = {"semana": 0, "fin_semana": 1}
df["tipo_dia_cod"] = df["tipo_dia"].map(map_tipo_dia)

map_categoria = {
    "hotel 3 estrellas": 1,
    "hotel 4 estrellas": 2,
    "hotel 5 estrellas": 3,
    "apartamento": 4,
    "casa entera": 5,
    "habitacion privada": 6,
    "habitacion compartida": 7,
    "alternativo": 8
}
df["categoria_alojamiento_cod"] = df["categoria_alojamiento"].map(map_categoria)

map_antelacion = {
    "1 semana": 1,
    "2 semanas": 2,
    "1 mes": 3,
    "2-3 meses": 4,
    "3 meses": 5
}
df["periodo_antelacion_cod"] = df["periodo_antelacion"].map(map_antelacion)

map_temporada = {"baja": 1, "media": 2, "alta": 3}
df["temporada_cod"] = df["temporada"].map(map_temporada)

df["es_fin_de_semana"] = df["tipo_dia_cod"]

columnas_finales = [
    "id_alojamiento",
    "id_pais",
    "id_ccaa",
    "id_provincia",
    "mes",
    "temporada",
    "temporada_cod",
    "categoria_alojamiento",
    "categoria_alojamiento_cod",
    "periodo_antelacion",
    "periodo_antelacion_cod",
    "valoraciones_norm",
    "tiene_valoraciones",
    "tipo_dia",
    "tipo_dia_cod",
    "es_fin_de_semana",
    "precio"
]

df = df[columnas_finales].copy()
df.to_csv(output_features_path, index=False, encoding="utf-8")

print("Dimensiones dataset features:", df.shape)
display(df.head())
display(df.isnull().sum())

Dimensiones dataset features: (20560, 17)


,id_alojamiento,id_pais,id_ccaa,id_provincia,mes,temporada,temporada_cod,categoria_alojamiento,categoria_alojamiento_cod,periodo_antelacion,periodo_antelacion_cod,valoraciones_norm,tiene_valoraciones,tipo_dia,tipo_dia_cod,es_fin_de_semana,precio
0,1,ES,1,4,1,baja,1,hotel 3 estrellas,1,1 mes,3,3.7595,True,semana,0,0,70.00
1,2,ES,1,4,1,baja,1,hotel 3 estrellas,1,1 semana,1,3.7595,True,semana,0,0,71.00
2,3,ES,1,4,1,baja,1,hotel 3 estrellas,1,2 semanas,2,3.7595,True,semana,0,0,71.00
3,4,ES,1,4,1,baja,1,hotel 3 estrellas,1,3 meses,5,3.7595,True,semana,0,0,67.33
4,5,ES,1,4,1,baja,1,hotel 4 estrellas,2,1 mes,3,3.2400,True,semana,0,0,75.33


id_alojamiento               0
id_pais                      0
id_ccaa                      0
id_provincia                 0
mes                          0
temporada                    0
temporada_cod                0
categoria_alojamiento        0
categoria_alojamiento_cod    0
periodo_antelacion           0
periodo_antelacion_cod       0
valoraciones_norm            0
tiene_valoraciones           0
tipo_dia                     0
tipo_dia_cod                 0
es_fin_de_semana             0
precio                       0
dtype: int64

El feature engineering transforma variables categóricas en representaciones numéricas utilizables por los algoritmos de regresión. Además, incorpora una variable temporal de temporada y una variable indicadora del tipo de día, relevantes para explicar la variación del precio.

En esta etapa se entrenan y comparan distintos modelos de regresión para seleccionar el que mejor predice el precio del alojamiento.

In [52]:
features = [
    "id_ccaa",
    "id_provincia",
    "mes",
    "temporada_cod",
    "categoria_alojamiento_cod",
    "periodo_antelacion_cod",
    "valoraciones_norm",
    "tiene_valoraciones",
    "tipo_dia_cod"
]

X = df[features]
y = df["precio"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Tamaño entrenamiento:", X_train.shape)
print("Tamaño prueba:", X_test.shape)

Tamaño entrenamiento: (16448, 9)
Tamaño prueba: (4112, 9)


In [53]:
modelos = {
    "LinearRegression": LinearRegression(),
    "RandomForestRegressor": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    "GradientBoostingRegressor": GradientBoostingRegressor(
        n_estimators=200,
        random_state=42
    )
}

resultados = []

for nombre_modelo, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    r2 = r2_score(y_test, y_pred)

    resultados.append({
        "modelo": nombre_modelo,
        "MAE": round(mae, 4),
        "RMSE": round(rmse, 4),
        "R2": round(r2, 4)
    })

df_resultados = pd.DataFrame(resultados).sort_values(by="R2", ascending=False)
df_resultados

,modelo,MAE,RMSE,R2
1,RandomForestRegressor,4.4545,11.9228,0.9823
2,GradientBoostingRegressor,25.7614,41.9583,0.7813
0,LinearRegression,58.7186,86.6425,0.0673


Para evaluar el desempeño de los modelos de regresión se utilizaron tres métricas principales: MAE (Mean Absolute Error), RMSE (Root Mean Squared Error) y R² (coeficiente de determinación). Estas métricas permiten medir la precisión de las predicciones del modelo respecto a los valores reales del precio del alojamiento.

El MAE representa el error absoluto promedio entre el valor real y el valor predicho. Esta métrica es fácilmente interpretable porque se expresa en las mismas unidades que la variable objetivo (precio). Un valor menor de MAE indica que el modelo comete errores de menor magnitud en promedio.

El RMSE mide la raíz del error cuadrático medio y penaliza con mayor intensidad los errores grandes. Por ello, esta métrica resulta útil para identificar si existen predicciones con desviaciones significativas respecto al valor real.

Por su parte, el coeficiente de determinación R² indica la proporción de la variabilidad del precio que puede ser explicada por las variables independientes incluidas en el modelo. Un valor cercano a 1 indica un mayor poder explicativo del modelo, mientras que valores más cercanos a 0 sugieren una capacidad limitada para explicar la variabilidad de los datos.

Tras comparar los diferentes algoritmos evaluados, el modelo Random Forest Regressor mostró el mejor desempeño global en términos de equilibrio entre error y capacidad explicativa. Este comportamiento es consistente con la naturaleza del problema, ya que los modelos basados en árboles suelen capturar mejor relaciones no lineales entre variables, así como interacciones complejas entre características del alojamiento, ubicación geográfica y factores temporales.

La selección de este modelo permite obtener estimaciones de precio más robustas, lo cual resulta especialmente relevante para el objetivo del proyecto CulturaTrip, donde se busca ofrecer estimaciones de presupuesto de viaje basadas en datos reales del mercado turístico.

En consecuencia, el modelo seleccionado se utilizará en la fase de simulación para estimar el coste del alojamiento en función de las características del viaje, permitiendo evaluar si el presupuesto del usuario es suficiente para el destino y periodo seleccionado.

In [54]:
metricas_path = ML_DIR / "metricas_modelos_regresion.csv"
df_resultados.to_csv(metricas_path, index=False, encoding="utf-8")
print(metricas_path)

C:\Users\Andrea Morales Vega\Downloads\TFM\CulturaTrip_TFM\data\Machine Learning\metricas_modelos_regresion.csv


In [55]:
mejor_modelo_nombre = df_resultados.iloc[0]["modelo"]
mejor_modelo = modelos[mejor_modelo_nombre]
mejor_modelo.fit(X_train, y_train)

if hasattr(mejor_modelo, "feature_importances_"):
    importancias = pd.DataFrame({
        "variable": features,
        "importancia": mejor_modelo.feature_importances_
    }).sort_values(by="importancia", ascending=False)
    display(importancias)

,variable,importancia
4,categoria_alojamiento_cod,0.356025
0,id_ccaa,0.246600
1,id_provincia,0.158657
6,valoraciones_norm,0.149005
2,mes,0.044366
3,temporada_cod,0.026131
8,tipo_dia_cod,0.013924
5,periodo_antelacion_cod,0.005291
7,tiene_valoraciones,0.000000


El modelo Random Forest permite analizar la relevancia relativa de cada variable en la predicción del precio del alojamiento mediante la métrica de feature importance. Esta métrica indica cuánto contribuye cada variable a reducir el error de predicción dentro de los árboles del modelo.

Los resultados muestran que la variable más influyente es categoria_alojamiento_cod, con una importancia aproximada del 35.6 %. Esto indica que el tipo de alojamiento (por ejemplo, hotel de distintas categorías, apartamento o casa completa) es el factor que más explica las diferencias de precio entre observaciones. Este resultado es coherente con la lógica del mercado turístico, donde la categoría del alojamiento suele determinar gran parte del coste final.

En segundo lugar aparece id_ccaa (24.6 %), seguido por id_provincia (15.9 %), lo que evidencia que la localización geográfica también tiene un impacto significativo en el precio. Esto refleja las diferencias estructurales de precios entre regiones turísticas, donde ciertos destinos presentan niveles de coste superiores debido a factores como demanda turística, infraestructura o atractivo del destino.

Otra variable relevante es valoraciones_norm (14.9 %), lo que sugiere que las valoraciones o reputación del alojamiento influyen en el precio esperado. En plataformas turísticas es habitual que alojamientos mejor valorados puedan mantener precios más elevados.

En contraste, variables temporales como mes (4.4 %) y temporada_cod (2.6 %) presentan una influencia menor. Esto puede deberse a que el dataset ya refleja precios agregados o que las diferencias estacionales están parcialmente capturadas por otras variables del modelo.

Por último, variables como tipo_dia_cod (diferencia entre semana y fin de semana) y periodo_antelacion_cod muestran una contribución relativamente baja, mientras que tiene_valoraciones no presenta impacto significativo en el modelo, probablemente porque la información relevante ya está representada en la variable continua de valoraciones.

En conjunto, los resultados sugieren que las características estructurales del alojamiento y su ubicación geográfica son los principales determinantes del precio, mientras que los factores temporales y operativos tienen un papel secundario dentro del modelo.

Estos resultados son especialmente relevantes para la plataforma CulturaTrip, ya que permiten identificar los factores que más influyen en el coste del alojamiento y facilitan la estimación de presupuestos de viaje personalizados en función del destino y del tipo de hospedaje seleccionado por el usuario.

Una vez seleccionado el mejor modelo, se entrena sobre el conjunto completo de datos y se persiste para su reutilización posterior en el simulador.

In [56]:
modelo_final = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

modelo_final.fit(X, y)

model_path = MODEL_DIR / "random_forest_precio.pkl"
features_path = MODEL_DIR / "features_modelo_precio.pkl"

with open(model_path, "wb") as archivo_modelo:
    pickle.dump(modelo_final, archivo_modelo)

with open(features_path, "wb") as archivo_features:
    pickle.dump(features, archivo_features)

print(model_path)
print(features_path)

C:\Users\Andrea Morales Vega\Downloads\TFM\CulturaTrip_TFM\outputs\regresion_precios\modelos\random_forest_precio.pkl
C:\Users\Andrea Morales Vega\Downloads\TFM\CulturaTrip_TFM\outputs\regresion_precios\modelos\features_modelo_precio.pkl


Finalmente, se valida el valor práctico del modelo mediante una simulación de presupuesto turístico.

In [57]:
with open(model_path, "rb") as archivo_modelo:
    modelo = pickle.load(archivo_modelo)

with open(features_path, "rb") as archivo_features:
    features_guardadas = pickle.load(archivo_features)

In [58]:
def calcular_noches(fecha_inicio, fecha_fin):
    fecha_inicio = datetime.strptime(fecha_inicio, "%Y-%m-%d")
    fecha_fin = datetime.strptime(fecha_fin, "%Y-%m-%d")

    noches = []
    fecha_actual = fecha_inicio

    while fecha_actual < fecha_fin:
        noches.append(fecha_actual)
        fecha_actual += timedelta(days=1)

    return noches

def separar_noches(noches):
    noches_semana = 0
    noches_fin_semana = 0

    for fecha in noches:
        if fecha.weekday() >= 4:
            noches_fin_semana += 1
        else:
            noches_semana += 1

    return noches_semana, noches_fin_semana

def predecir_precio(base_input):
    df_input = pd.DataFrame([base_input])
    return modelo.predict(df_input)[0]

In [61]:
usuario = {
    "presupuesto": 8000,
    "fecha_inicio": "2025-07-10",
    "fecha_fin": "2025-07-15",
    "id_ccaa": 1,
    "id_provincia": 29,
    "mes": 7,
    "temporada_cod": 3,
    "categoria_alojamiento_cod": 2,
    "periodo_antelacion_cod": 3,
    "valoraciones_norm": 4.2,
    "tiene_valoraciones": True
}

pct_alojamiento = 0.35
budget_alojamiento = usuario["presupuesto"] * pct_alojamiento

noches = calcular_noches(usuario["fecha_inicio"], usuario["fecha_fin"])
noches_semana, noches_fin_semana = separar_noches(noches)

input_semana = {
    "id_ccaa": usuario["id_ccaa"],
    "id_provincia": usuario["id_provincia"],
    "mes": usuario["mes"],
    "temporada_cod": usuario["temporada_cod"],
    "categoria_alojamiento_cod": usuario["categoria_alojamiento_cod"],
    "periodo_antelacion_cod": usuario["periodo_antelacion_cod"],
    "valoraciones_norm": usuario["valoraciones_norm"],
    "tiene_valoraciones": usuario["tiene_valoraciones"],
    "tipo_dia_cod": 0
}

input_fin_semana = input_semana.copy()
input_fin_semana["tipo_dia_cod"] = 1

precio_semana = predecir_precio(input_semana)
precio_fin_semana = predecir_precio(input_fin_semana)

coste_total = noches_semana * precio_semana + noches_fin_semana * precio_fin_semana

print("Presupuesto total:", round(usuario["presupuesto"], 2))
print("Presupuesto disponible para alojamiento:", round(budget_alojamiento, 2))
print("Noches semana:", noches_semana)
print("Noches fin de semana:", noches_fin_semana)
print("Precio estimado semana:", round(precio_semana, 2))
print("Precio estimado fin semana:", round(precio_fin_semana, 2))
print("Coste total estimado alojamiento:", round(coste_total, 2))

if budget_alojamiento >= coste_total:
    print("El presupuesto de alojamiento ALCANZA para este viaje.")
else:
    print("El presupuesto de alojamiento NO alcanza para este viaje.")

Presupuesto total: 8000
Presupuesto disponible para alojamiento: 2800.0
Noches semana: 2
Noches fin de semana: 3
Precio estimado semana: 87.23
Precio estimado fin semana: 103.58
Coste total estimado alojamiento: 485.19
El presupuesto de alojamiento ALCANZA para este viaje.


La simulación traduce el resultado del modelo a un caso de uso comprensible para el usuario final, estimando el coste del hospedaje y comparándolo con un presupuesto disponible.